# Sign Language MNIST — Image Classification Using a CNN

**Assignment:** Train a convolutional neural network on 28 × 28 grayscale images of American Sign Language hand signs.

**Required architecture (in order):** `Conv2D(32, 3×3, relu) → MaxPooling2D → Conv2D(64, 3×3, relu) → MaxPooling2D → Flatten → Dense(64, relu) → Dense(n_classes, softmax)`.

**Deliverables produced when you run this notebook:**
1. A trained CNN using **5 epochs** and a printed **test accuracy**.
2. A **training vs. validation accuracy** plot.
3. A grid of **5 test images**, each labeled with the true and predicted letter.

**Dataset:** [Sign Language MNIST (Kaggle)](https://www.kaggle.com/datasets/datamunge/sign-language-mnist). Each CSV row has one `label` plus **784 pixel values**. The dataset contains **24 static letters** (J and Z are omitted).

> **Before running:** Make `sign_mnist_train.csv` and, ideally, `sign_mnist_test.csv` available in the notebook folder or `data/`. Kaggle's `/kaggle/input/` folder is also searched automatically. If the test CSV is missing, the notebook creates a held-out test set from the training CSV. No accuracy is pre-filled: the notebook calculates it from your own run.

## 1. Import libraries and fix random seeds

TensorFlow/Keras handles the CNN; scikit-learn handles data splitting and label encoding. If a library is missing, run `%pip install tensorflow pandas numpy matplotlib scikit-learn` in a separate cell first.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

SEED = 42
SAMPLE_SIZE = 5_000      # Randomly sample up to 5,000 training rows
EPOCHS = 5               # Exactly 5, as requested
BATCH_SIZE = 64

np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)

## 2. Load the CSV file(s)

Use the published train/test split if both CSVs exist. Otherwise, reserve 20% of the sampled rows for testing. In either case, a **separate validation set** will be used during training.

In [ ]:
def find_csv(filename):
    """Find a Sign Language MNIST CSV in common local, Colab, or Kaggle locations."""
    for candidate in [
        Path(filename),
        Path("data") / filename,
        Path("/content") / filename,
        Path("/content/data") / filename,
    ]:
        if candidate.is_file():
            return candidate

    for directory in [Path.cwd(), Path("/kaggle/input")]:
        if directory.is_dir():
            matches = sorted(directory.rglob(filename))
            if matches:
                return matches[0]
    return None

train_path = find_csv("sign_mnist_train.csv")
test_path = find_csv("sign_mnist_test.csv")

if train_path is None:
    raise FileNotFoundError(
        "Cannot find sign_mnist_train.csv. Download the Sign Language MNIST "
        "dataset and put its CSV files beside this notebook or in data/."
    )

full_train_df = pd.read_csv(train_path)
if "label" not in full_train_df.columns:
    raise ValueError("The training CSV must have a 'label' column.")
if full_train_df.shape[1] != 785:
    raise ValueError(f"Expected 785 columns (label + 784 pixels), found {full_train_df.shape[1]}.")

# Exactly the requested random 5,000-row sample (or all rows if there are fewer).
sampled_df = full_train_df.sample(
    n=min(SAMPLE_SIZE, len(full_train_df)), random_state=SEED
).reset_index(drop=True)

if test_path is not None:
    train_pool_df = sampled_df
    test_df = pd.read_csv(test_path)
    if set(test_df.columns) != set(full_train_df.columns):
        raise ValueError("Training and test CSV files must have the same columns.")
    test_df = test_df[full_train_df.columns].reset_index(drop=True)
    print("Using the provided independent test CSV:", test_path)
else:
    train_pool_df, test_df = train_test_split(
        sampled_df, test_size=0.20, stratify=sampled_df["label"], random_state=SEED
    )
    print("No test CSV found: using a stratified 20% held-out test split.")

print("Training source:", train_path)
print(f"Original training CSV rows: {len(full_train_df):,}")
print(f"Sampled rows: {len(sampled_df):,}")
print(f"Rows available for train + validation: {len(train_pool_df):,}")
print(f"Test rows: {len(test_df):,}")
display(sampled_df.head(3).iloc[:, :9])  # Show label plus first 8 pixels

## 3. Separate labels, split training/validation, reshape, and normalize

- `label` is the target; the other **784 columns** are grayscale pixel intensities.
- Each row reshapes from `(784,)` to `(28, 28, 1)`.
- Dividing by `255.0` converts pixels to the **0–1 range**.
- `LabelEncoder` maps original label IDs to consecutive `0, …, n_classes−1` indices; this is useful because Sign Language MNIST skips some alphabet IDs.

In [ ]:
train_df, val_df = train_test_split(
    train_pool_df, test_size=0.20,
    stratify=train_pool_df["label"], random_state=SEED
)

# Fit on labels from the original training file, not from the test set.
label_encoder = LabelEncoder().fit(full_train_df["label"])
n_classes = len(label_encoder.classes_)

# With the standard dataset, all test labels occur in the training dataset.
unknown_labels = set(test_df["label"]) - set(label_encoder.classes_)
if unknown_labels:
    raise ValueError(f"Test set contains unseen class labels: {sorted(unknown_labels)}")

def prepare_images(frame):
    pixels = frame.drop(columns="label").to_numpy(dtype=np.float32)
    if pixels.shape[1] != 784:
        raise ValueError("Each row must contain exactly 784 pixel columns.")
    return pixels.reshape(-1, 28, 28, 1) / 255.0

def prepare_labels(frame):
    return label_encoder.transform(frame["label"]).astype(np.int32)

X_train, y_train = prepare_images(train_df), prepare_labels(train_df)
X_val, y_val = prepare_images(val_df), prepare_labels(val_df)
X_test, y_test = prepare_images(test_df), prepare_labels(test_df)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)
print("Pixel range:", float(X_train.min()), "to", float(X_train.max()))
print("Number of classes:", n_classes)
print("Original class IDs:", label_encoder.classes_.tolist())

assert X_train.shape[1:] == (28, 28, 1)
assert 0.0 <= X_train.min() <= X_train.max() <= 1.0

## 4. Define the **exact requested CNN**

No extra convolution, dropout, or batch-normalization layers are added. Max-pooling uses Keras's default `(2, 2)` pool size.

In [ ]:
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(n_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 5. Train the model for **5 epochs**

Validation accuracy is measured after each epoch using the validation subset; the **test set is not used for training or validation**.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

## 6. Report test accuracy

This is the result to write in your assignment submission. The exact percentage depends on the random sample, environment, and training run.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

## 7. Plot training vs. validation accuracy

The two curves help show learning progress and whether the model begins to overfit.

In [ ]:
epochs_ran = range(1, len(history.history["accuracy"]) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs_ran, history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.title("Sign Language MNIST CNN: Training vs. Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(list(epochs_ran))
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Display **5 sample test images** with true and predicted labels

The original numeric class IDs map to letters of the alphabet: `0 → A`, `1 → B`, etc. J and Z are excluded from this static-sign dataset.

In [ ]:
# Predict class probabilities and pick the most likely class for each test image.
class_probabilities = model.predict(X_test, verbose=0)
y_pred = np.argmax(class_probabilities, axis=1)

def readable_label(encoded_class):
    original_label = int(label_encoder.inverse_transform([int(encoded_class)])[0])
    letter = chr(ord("A") + original_label)
    return f"{letter} ({original_label})"

rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(X_test), size=5, replace=False)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, idx in zip(axes, sample_indices):
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray")
    true_name = readable_label(y_test[idx])
    predicted_name = readable_label(y_pred[idx])
    verdict = "Correct" if y_test[idx] == y_pred[idx] else "Incorrect"
    ax.set_title(f"True: {true_name}\nPred: {predicted_name}\n{verdict}", fontsize=10)
    ax.axis("off")

fig.suptitle("Five Random Sign Language MNIST Test Predictions", y=1.08)
plt.tight_layout()
plt.show()

## Submission checklist

- [x] Load Sign Language MNIST CSV and take a reproducible 5,000-row random training sample.
- [x] Separate the `label` column and reshape pixels to `(28, 28, 1)`.
- [x] Normalize pixel values by dividing by `255`.
- [x] Implement the **exact** CNN specified in the problem.
- [x] Train for **5 epochs**.
- [x] Include a cell that prints **test accuracy**.
- [x] Include training and validation accuracy plot code.
- [x] Include a grid of five test predictions with true and predicted labels.

**Final submission:** Run **Kernel → Restart & Run All** (or **Runtime → Run all** in Colab/Kaggle), check that the accuracy and both figures display, and save/download the executed `.ipynb` file so the outputs are retained.